# 08_first_vqe_end_to_end

**Purpose:**

- Run a complete Variational Quantum Eigensolver (VQE) workflow once
- Understand VQE as a variational principle, not a recipe
- Identify the quantum vs classical roles in the algorithm

**Core idea:**

VQE minimizes the energy expectation value  
⟨ψ(θ)| H |ψ(θ)⟩  
over a *restricted, parameterized family of quantum states* defined by an ansatz.

In this notebook, expectation values are evaluated exactly using statevector simulation (either explicitly or via `StatevectorEstimator`). On real hardware, these expectation values are instead estimated from noisy quantum measurements and combined with a classical optimizer.

In [1]:
# ----- Manual VQE loop (statevector + SciPy optimizer) -----

# Minimal VQE (Variational Quantum Eigensolver) loop for a 2-qubit toy Hamiltonian.
# Core idea (variational principle): minimize E(θ) = <ψ(θ)| H |ψ(θ)>
# over a restricted family of states |ψ(θ)> produced by an ansatz circuit.

import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

# --- 1) Problem definition: choose a Hamiltonian H (2 qubits) ---
# H = 1.0 * (Z ⊗ Z) + 0.5 * (X ⊗ I) + 0.5 * (I ⊗ X)
H = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

# --- 2) Ansatz definition (quantum part): a small hardware-efficient circuit family ---
# Here: local RY rotations + one entangling CX gate.
def ansatz(params: np.ndarray) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    qc.cx(0, 1)
    return qc

# --- 3) Energy evaluation E(θ) = <ψ(θ)|H|ψ(θ)> ---
# This version uses exact statevector simulation (no shot noise / sampling).
H_mat = H.to_matrix()

def energy(params: np.ndarray) -> float:
    psi = Statevector.from_instruction(ansatz(params))
    return float(np.real(np.vdot(psi.data, H_mat @ psi.data)))

# --- 4) Classical optimization (classical part): choose θ to minimize E(θ) ---
x0 = np.array([0.1, 0.2], dtype=float)

print("Energy at initial point:", energy(x0))

try:
    # Prefer a standard optimizer when SciPy is available.
    from scipy.optimize import minimize
    res = minimize(energy, x0, method="COBYLA", options={"maxiter": 100})
    print("Estimated ground-state energy:", res.fun)
    print("Best parameters:", res.x)
    print("Optimizer success:", res.success, "|", res.message)
except Exception as e:
    # Minimal fallback: random search (not efficient, but demonstrates the VQE loop).
    print("SciPy not available; falling back to random search. Error:", e)
    best_E = float("inf")
    best_x = None
    rng = np.random.default_rng(0)
    for _ in range(300):
        x = rng.uniform(0, 2*np.pi, size=2)
        E = energy(x)
        if E < best_E:
            best_E, best_x = E, x
    print("Estimated ground-state energy (random search):", best_E)
    print("Best parameters:", best_x)

Energy at initial point: 1.0893181622768775
Estimated ground-state energy: -1.0000019772444617
Best parameters: [-1.48157648  3.1435582 ]
Optimizer success: False | Maximum number of function evaluations has been exceeded.


In [2]:
# ----- Qiskit VQE class (StatevectorEstimator / optional Aer) -----

# Minimal VQE (Variational Quantum Eigensolver) on a 2-qubit Hamiltonian.
# Goal: run VQE end-to-end and print an estimated ground-state energy.
#
# This cell defaults to an *exact* statevector-based estimator (deterministic, no shot noise),
# and optionally uses an Aer-based estimator if it is available AND compatible.

import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import TwoLocal

# --- 1) Define a simple 2-qubit Hamiltonian H ---
# H = 1.0 * (Z ⊗ Z) + 0.5 * (X ⊗ I) + 0.5 * (I ⊗ X)
H = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

# --- 2) Define a small hardware-efficient ansatz (parameterized circuit family) ---
# Note: TwoLocal is deprecated in Qiskit 2.1+ but still works. We keep it for minimal first exposure.
ansatz = TwoLocal(
    num_qubits=2,
    rotation_blocks="ry",
    entanglement_blocks="cx",
    reps=1,
)

# IMPORTANT (Aer compatibility): expand the template into primitive gate instructions
ansatz = ansatz.decompose()

# Debug: verify the template was expanded into primitive gates (Aer compatibility)
print(ansatz.count_ops())

# --- 3) Import VQE + optimizer from qiskit_algorithms ---
from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA

optimizer = COBYLA(maxiter=100)

# --- 4) Choose an Estimator primitive ---
# Prefer Aer if it is available and compatible; otherwise fall back to StatevectorEstimator,
# which is exact and tends to be the most version-stable option.
using = None
estimator = None

# (A) Try Aer-based estimator (optional acceleration / closer-to-runtime primitives)
try:
    # Use Aer EstimatorV2 (VQE expects the V2 primitives interface)
    from qiskit_aer.primitives import EstimatorV2 as AerEstimatorV2  # type: ignore
    estimator = AerEstimatorV2()
    using = "qiskit_aer.primitives.EstimatorV2"
except Exception:
    estimator = None
    using = None

# (B) Fallback: exact statevector estimator (robust baseline)
if estimator is None:
    from qiskit.primitives import StatevectorEstimator
    estimator = StatevectorEstimator()
    using = "qiskit.primitives.StatevectorEstimator (exact)"
    
# --- 5) Run VQE: classical optimizer proposes θ, quantum estimator evaluates ⟨H⟩ ---
vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)
result = vqe.compute_minimum_eigenvalue(H)

print("Estimator backend:", using)
print("Estimated ground-state energy:", float(np.real(result.eigenvalue)))

OrderedDict({'ry': 4, 'cx': 1})


/var/folders/s1/t5h9_47x4357q7sl4ykbgmrc0000gn/T/ipykernel_66277/2384194691.py:21: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(


Estimator backend: qiskit_aer.primitives.EstimatorV2
Estimated ground-state energy: -1.3428549639401688


### What VQE did here

- The Hamiltonian defines the problem (target operator).
- The ansatz defines the allowed family of states.
- The quantum backend evaluates ⟨H⟩ (exactly here via statevector simulation; via noisy measurements on hardware).
- The classical optimizer updates parameters to reduce energy.

Nothing guarantees the exact ground state — only the best state reachable by the ansatz. 

### Key insight:

The VQE's success is limited by the expressivity of the ansatz and the noise tolerance of the hardware, not by the optimizer.

In [3]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import TwoLocal

from qiskit_algorithms.minimum_eigensolvers import VQE
from qiskit_algorithms.optimizers import COBYLA, SPSA

# --- Hamiltonian (same as earlier) ---
H = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

# --- Estimator (reuse the existing estimator if already defined) ---
# If estimator is not already defined in the notebook, fall back to StatevectorEstimator here.
try:
    estimator
except NameError:
    from qiskit.primitives import StatevectorEstimator
    estimator = StatevectorEstimator()

def run_vqe(reps: int, optimizer, label: str) -> float:
    """Run VQE for a given ansatz depth and optimizer; return the final energy."""
    ansatz = TwoLocal(
        num_qubits=2,
        rotation_blocks="ry",
        entanglement_blocks="cx",
        reps=reps,
    ).decompose()  # helps avoid Aer seeing 'TwoLocal' as an unknown instruction

    vqe = VQE(estimator=estimator, ansatz=ansatz, optimizer=optimizer)
    result = vqe.compute_minimum_eigenvalue(H)
    energy = float(np.real(result.eigenvalue))
    print(f"{label}: reps={reps}, optimizer={optimizer.__class__.__name__}, energy={energy}")
    return energy

# --- 1) Baseline: reps=1, COBYLA ---
E_baseline = run_vqe(reps=1, optimizer=COBYLA(maxiter=100), label="Baseline")

# --- 2) More expressive ansatz: reps=2, COBYLA ---
E_reps2 = run_vqe(reps=2, optimizer=COBYLA(maxiter=100), label="Ansatz deeper")

# --- 3) Same ansatz depth, different optimizer: reps=2, SPSA ---
E_spsa = run_vqe(reps=2, optimizer=SPSA(maxiter=100), label="Optimizer change (SPSA)")

print("\nSummary (lower is better):")
print("Baseline (reps=1, COBYLA):", E_baseline)
print("Deeper ansatz (reps=2, COBYLA):", E_reps2)
print("Deeper ansatz + SPSA        :", E_spsa)

/var/folders/s1/t5h9_47x4357q7sl4ykbgmrc0000gn/T/ipykernel_66277/3469196922.py:25: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(
/var/folders/s1/t5h9_47x4357q7sl4ykbgmrc0000gn/T/ipykernel_66277/3469196922.py:25: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(
/var/folders/s1/t5h9_47x4357q7sl4ykbgmrc0000gn/T/ipykernel_66277/3469196922.py:25: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(


Baseline: reps=1, optimizer=COBYLA, energy=-1.4142135559657827
Ansatz deeper: reps=2, optimizer=COBYLA, energy=-1.414213138212208
Optimizer change (SPSA): reps=2, optimizer=SPSA, energy=-1.4133242991064652

Summary (lower is better):
Baseline (reps=1, COBYLA): -1.4142135559657827
Deeper ansatz (reps=2, COBYLA): -1.414213138212208
Deeper ansatz + SPSA        : -1.4133242991064652


### Interpretation of the modification results

- The best energies from COBYLA are extremely close to \(-\sqrt{2}\), suggesting the reps=1 ansatz is already expressive enough for this toy Hamiltonian.
- Increasing ansatz depth (reps=2) did not materially improve the result, consistent with having already reached the variational optimum within optimizer tolerance.
- SPSA performed slightly worse in this statevector (noise-free) setting. This is expected: SPSA is primarily advantageous when energy estimates are noisy (shot noise / hardware noise), whereas here the objective is deterministic.